# E-Commerce Product Classifier — Hinglish Multi-Model Weighted Voting

**Pipeline:** Hinglish text → Normalize → Tokenize → BERT + LSTM + ML Ensemble → Weighted Soft Voting → Label + Confidence

```
Input (noisy Hinglish)
       ↓
  Normalizer  (informal spellings, code-mix)
       ↓
  BPE Tokenizer (TikTok-style)
       ↓
  ┌────────────┬──────────────┬──────────────┐
  │  BERT      │  BiLSTM      │  LR+RF+XGB   │
  │  (0.60)    │  (0.25)      │  (0.15)      │
  └─────┬──────┴──────┬───────┴──────┬───────┘
         └────────────┼──────────────┘
                Weighted Soft Vote
                       ↓
              Label + Confidence Score
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
!pip install torch transformers tokenizers scikit-learn xgboost seaborn --quiet
print('Setup complete')

## 1. Dataset — Hinglish Product Descriptions

In [ ]:
from data.sample_data import get_dataframe, LABEL2ID, ID2LABEL
import pandas as pd

df = get_dataframe()
print(f'Total samples: {len(df)}')
print(df['label'].value_counts())
df.head(10)

## 2. Preprocessing — Normalizer

In [ ]:
from preprocessing.normalizer import normalize

examples = [
    'saari cotton printedddd floral',
    'jwellery necklace kundan bridal',
    'mobail phone 5g android new best',
    'mehandi cone natural dark free offer',
]
for ex in examples:
    print(f'  Before: {ex}')
    print(f'  After : {normalize(ex)}')
    print()

## 3. BPE Tokenizer (TikTok-style)

In [ ]:
from preprocessing.normalizer import normalize_batch
from preprocessing.tokenizer import BPETokenizer

texts_clean = normalize_batch(df['text'].tolist())
bpe = BPETokenizer(vocab_size=4000, save_path='bpe_tokenizer.json')
bpe.train(texts_clean)

sample = 'saree pin gold fancy'
print(f'Input  : {sample}')
print(f'Tokens : {bpe.encode(normalize(sample))}')

## 4. Train All Models

In [ ]:
from train import main
results = main()

## 4b. Clear Cache After Training

In [ ]:
from utils.cache_cleaner import clean
clean()  # frees GPU VRAM, closes figures, runs GC, reloads module cache

## 5. Model Comparison Chart

In [ ]:
from evaluation.metrics import compare_models
compare_models(results)

## 6. Inference — Classify New Products

In [ ]:
from predict import predict

test_cases = [
    'saree pin gold fancy',           # jewellery NOT clothing
    'banarasi silk saree red border',  # clothing
    'mobile phone 5g android',         # mobile_accessories
    'kajal black waterproof',          # beauty
    'diya clay diwali set',            # home_decor
    'jutti punjabi embroidered',       # footwear
    'sofa set 3 seater fabric',        # home_furniture
    'sunglasses polarized men',        # eyewear
    'watch men analog leather',        # watches
    'laptop 15 inch i5 8gb',           # laptops
    'mixer grinder 750w 3 jar',        # kitchen_appliances
    'basmati rice 5kg premium',        # food_supplies
    'seeds tomato hybrid packet',      # agriculture
    'acid hydrochloric lab grade',     # hazardous
    'notebook spiral a4 200 pages',    # stationery
    'printer inkjet color home',       # printers
    'jeans slim fit blue denim',       # garments
    'cricket bat junior size',         # sportswear
    'saari cotton printed',            # clothing (informal spelling)
    'jwellery necklace kundan',        # jewellery (misspelled)
]

results_inf = predict(test_cases)
for text, res in zip(test_cases, results_inf):
    bar = '█' * int(res['confidence'] * 20)
    print(f'{text:<42} → {res["label"]:<20} {bar} {res["confidence"]*100:.1f}%')

## 7. Confusion Matrix

In [ ]:
from IPython.display import Image
Image('confusion_matrix.png')

## 8. Weighted Voting — How It Works

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Hardcoded — no import dependency, always works
labels = [
    'clothing', 'jewellery', 'beauty', 'footwear', 'home_decor',
    'home_furniture', 'eyewear', 'watches', 'mobile_accessories', 'sportswear',
    'food_supplies', 'agriculture', 'hazardous', 'electronics', 'stationery',
    'kitchen_appliances', 'laptops', 'printers', 'garments'
]
n = len(labels)

def make_probs(clothing_score, jewellery_score):
    np.random.seed(None)
    p = np.abs(np.random.normal(0.02, 0.01, n))
    p[0] = clothing_score
    p[1] = jewellery_score
    return p / p.sum()

np.random.seed(42)
bert_p   = make_probs(0.05, 0.82)
lstm_p   = make_probs(0.55, 0.28)
ml_p     = make_probs(0.48, 0.32)
weighted = 0.60*bert_p + 0.25*lstm_p + 0.15*ml_p

x = np.arange(n)
fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)
fig.suptitle('Ambiguous Case: "saree pin gold" — How BERT saves the vote', fontweight='bold')

for ax, probs, title, color in zip(
    axes,
    [bert_p, lstm_p, ml_p, weighted],
    ['BERT (w=0.60)', 'LSTM (w=0.25)', 'ML (w=0.15)', 'Weighted Vote'],
    ['#3498db', '#e74c3c', '#27ae60', '#8e44ad']
):
    bars = ax.bar(x, probs, color=color, alpha=0.85, width=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=65, ha='right', fontsize=7)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1)
    pred_idx = int(probs.argmax())
    bars[pred_idx].set_edgecolor('gold')
    bars[pred_idx].set_linewidth(3)

plt.tight_layout()
plt.savefig('voting_visualization.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'Final prediction: {labels[int(weighted.argmax())]}.upper() ✅')